# Visualize Sampled Neurons

Loads a `.smol` file produced by `semlaflow/sample_neurons.py` (a `GeometricMolBatch` of raw predicted neuron graphs: coords + binary-edge adjacency) and renders each sample in interactive 3D.

**Dependencies.** Needs `plotly` and `ipywidgets` on top of the base env. If missing:
```
pip install plotly ipywidgets
```
For classic notebook, also run `jupyter nbextension enable --py widgetsnbextension` once.

In [ ]:
import sys
sys.path.append("..")

from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from ipywidgets import IntSlider, IntText, HBox, VBox, jslink
from IPython.display import display

import semlaflow.scriptutil as util
from semlaflow.util.molrepr import GeometricMolBatch
from semlaflow.data.swc import NEURON_EDGE_CLASS_INDEX

## Config

Point `SMOL_PATH` at the file written by `sample_neurons.py` (default: `<save_dir>/neuron_samples.smol`).

In [ ]:
SMOL_PATH = Path("../neuron_samples.smol")
COORDS_STD = util.NEURON_COORDS_STD_DEV  # coords in the .smol are standardised; multiply to recover physical units

assert SMOL_PATH.exists(), f"Could not find {SMOL_PATH.resolve()} — edit SMOL_PATH above."

## Load the batch

In [ ]:
batch = GeometricMolBatch.from_bytes(SMOL_PATH.read_bytes())
mols = batch.to_list()
print(f"Loaded {len(mols)} neuron samples from {SMOL_PATH}")

ex = mols[0]
print(f"Example sample 0: coords {tuple(ex.coords.shape)}, "
      f"bond_indices {tuple(ex.bond_indices.shape)}, "
      f"bond_types {tuple(ex.bond_types.shape)}")

## Summary stats

In [ ]:
node_counts = np.array([m.coords.shape[0] for m in mols])
edge_counts = np.array([m.bond_indices.shape[0] for m in mols])

print(f"Nodes per sample — min {node_counts.min()}, mean {node_counts.mean():.1f}, max {node_counts.max()}")
print(f"Edges per sample — min {edge_counts.min()}, mean {edge_counts.mean():.1f}, max {edge_counts.max()}")

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].hist(node_counts, bins=min(30, max(5, len(mols) // 2 or 1)))
axes[0].set_title("Nodes per sample")
axes[0].set_xlabel("# nodes")
axes[1].hist(edge_counts, bins=min(30, max(5, len(mols) // 2 or 1)))
axes[1].set_title("Predicted edges per sample")
axes[1].set_xlabel("# edges")
plt.tight_layout()
plt.show()

## Helper: build plotly traces for a single neuron

Edges are drawn as a single `Scatter3d` trace with `None` separators between segments (much faster than one trace per edge). `bond_indices` is already deduplicated upper-triangle by `samples_to_mols`, so we can plot them directly.

In [ ]:
def neuron_traces(mol, unscale: bool = True):
    coords = mol.coords.detach().cpu().numpy().astype(float)
    if unscale:
        coords = coords * COORDS_STD

    bonds = mol.bond_indices.detach().cpu().numpy().astype(int)

    node_trace = go.Scatter3d(
        x=coords[:, 0], y=coords[:, 1], z=coords[:, 2],
        mode="markers",
        marker=dict(size=3, color="#1f77b4"),
        name="nodes",
        hovertemplate="node %{text}<br>x=%{x:.2f}<br>y=%{y:.2f}<br>z=%{z:.2f}<extra></extra>",
        text=[str(i) for i in range(coords.shape[0])],
    )

    if bonds.shape[0] > 0:
        xs, ys, zs = [], [], []
        for a, b in bonds:
            xs.extend([coords[a, 0], coords[b, 0], None])
            ys.extend([coords[a, 1], coords[b, 1], None])
            zs.extend([coords[a, 2], coords[b, 2], None])
        edge_trace = go.Scatter3d(
            x=xs, y=ys, z=zs,
            mode="lines",
            line=dict(color="#444", width=2),
            name="edges",
            hoverinfo="skip",
        )
        return [edge_trace, node_trace]
    return [node_trace]

## Interactive viewer

Use the slider (or type into the index box) to scrub through every sample in the batch. Rotate / zoom with the mouse.

In [ ]:
viewer = go.FigureWidget(
    layout=dict(
        scene=dict(aspectmode="data",
                   xaxis_title="x", yaxis_title="y", zaxis_title="z"),
        height=650,
        margin=dict(l=0, r=0, t=40, b=0),
        showlegend=False,
    )
)

slider = IntSlider(min=0, max=len(mols) - 1, value=0, description="Sample", continuous_update=False)
index_box = IntText(value=0, description="Index")
jslink((slider, "value"), (index_box, "value"))


def _render(idx: int):
    idx = int(np.clip(idx, 0, len(mols) - 1))
    m = mols[idx]
    traces = neuron_traces(m)
    with viewer.batch_update():
        viewer.data = ()
        for t in traces:
            viewer.add_trace(t)
        viewer.layout.title = (
            f"Sample {idx} — {m.coords.shape[0]} nodes, {m.bond_indices.shape[0]} edges"
        )


_render(0)
slider.observe(lambda change: _render(change["new"]), names="value")

display(VBox([HBox([slider, index_box]), viewer]))

## Static figure fallback

If `ipywidgets` isn't set up (or you're exporting to HTML), render any single sample with a plain `go.Figure`.

In [ ]:
IDX = 0

m = mols[IDX]
static_fig = go.Figure(data=neuron_traces(m))
static_fig.update_layout(
    scene=dict(aspectmode="data", xaxis_title="x", yaxis_title="y", zaxis_title="z"),
    height=650,
    margin=dict(l=0, r=0, t=40, b=0),
    showlegend=False,
    title=f"Sample {IDX} — {m.coords.shape[0]} nodes, {m.bond_indices.shape[0]} edges",
)
static_fig.show()